In [1]:
from xtquant import xtdatacenter as xtdc
from xtquant import xtdata  
import pandas as pd 
import random
import datetime
import tushare as ts
import os
import yaml
def load_config():
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)
    return config
config = load_config()
TUSHARE_API_KEY = config["api"]["tushare_api_key"]
XTQUANT_API_KEY = config["api"]["xtquant_api_key"]
ts.set_token(TUSHARE_API_KEY)
pro = ts.pro_api(TUSHARE_API_KEY)
# xtdc.set_token("857b1fa0068599ccb191546f86bac28e09d8a727") # 试用过期
xtdc.set_token(XTQUANT_API_KEY)
xtdc.set_data_home_dir("../xtquant_tushare/srv/xuntoudata/data")
xtdc.init(False)
actual_port = random.randint(58600,58700)
xtdc.listen(port=actual_port)
print(f"服务启动,开放端口：{actual_port}")

服务启动,开放端口：58665


In [ ]:
# 初始模拟股票交易账户
from calendar import c
from real_time_index import read_last_trade_day_data_for_prepare_to_get_realtime_basis,get_realtime_computing_index
class TradingAccount:
    def __init__(self, account_id, initial_balance):
        """
        初始化交易账户
        :param account_id: 账号(str)
        :param initial_balance: 初始资金(float)
        """
        self.account_id = account_id
        self.initial_balance = initial_balance
        self.outside_balance = initial_balance  # 仓外余额
        self.inside_market_value = 0.0  # 仓内市值
        self.futures_balance = 0.0  # 期货仓余额
        self.futures_margin = {}  # 被冻结保证金
        self.total_asset = initial_balance  # 总资产
        self.holdings = {}  # 持仓，键为商品代码，值为字典  
    def get_current_price(self, symbol):
        # 获取商品当前市价
        if isinstance(symbol, str):
            real_time_stock_data = xtdata.get_full_tick([symbol])
            real_time_stock_lastPrice  = float('{:.3f}'.format(real_time_stock_data[symbol]['lastPrice']))
            return real_time_stock_lastPrice
        if isinstance(symbol, list):
            real_time_stock_data = xtdata.get_full_tick(symbol)
            real_time_stock_lastPrice = {stock: real_time_stock_data[stock]['lastPrice'] for stock in symbol}
            return real_time_stock_lastPrice
    def get_instrument_type(self, symbol):
        # 获取商品类型
        key_list= " ".join(xtdata.get_instrument_type(symbol).keys())
        type = ""
        if "STOCK" in key_list: # 股票
            type = "STOCK"
        elif "FUND" in key_list: # 基金
            type = "FUND"
        elif "INDEX" in key_list: # 指数
            type = "INDEX"
        else:
            type = "FUTURES"    # 期货
        return type   
    def get_instructment_details(self, symbol): # 获取商品详情
        return xtdata.get_instrument_detail(symbol, iscomplete=True)
    def open_a_futures_position(self,symbol,quantity,direction='go_long'): # 期货开仓，默认多头
        futures_data = self.get_instructment_details(symbol)
        # 提取合约相关信息
        ExpireDate = futures_data['ExpireDate']             # 合约到期日期
        LongMarginRatio = futures_data['LongMarginRatio']   # 多头保证金比率
        ShortMarginRatio = futures_data['ShortMarginRatio'] # 空头保证金比率
        VolumeMultiple = futures_data['VolumeMultiple']     # 合约乘数
        ChargeType = futures_data['ChargeType']             # 手续费收费方式
        ChargeOpen = futures_data['ChargeOpen']             # 开仓手续费
        ChargeClose = futures_data['ChargeClose']           # 平仓手续费
        ChargeTodayOpen = futures_data['ChargeTodayOpen']   # 今日开仓手续费
        ChargeTodayClose = futures_data['ChargeTodayClose'] # 今日平仓手续费
        # 计算保证金
        current_price = self.get_current_price(symbol)
        contract_value = current_price * quantity * VolumeMultiple
        
        commission = 0
        if ChargeTodayOpen != -1: # -1表示没有开仓手续费
            if ChargeType == 2: # 按费率收费
                commission = (ChargeTodayOpen / 10000) * contract_value
            elif ChargeType == 1: # 按手数收费
                commission = ChargeTodayOpen * quantity
            else:
                print("未知手续费收费方式")
        
        Margin = LongMarginRatio * contract_value 
        risk = Margin / (self.futures_balance - sum(self.futures_margin.values())) 
        
        if Margin < self.futures_balance - sum(self.futures_margin.values()):
            # 开仓成功
            self.futures_balance -= commission
            current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            if f"{symbol}_{direction}" in self.holdings and self.holdings[f"{symbol}_{direction}"]["trading_direction"] == direction: # 确保仓内是否已经存在多头合约
                self.futures_margin[f"{symbol}_{direction}"] += Margin
                holding = self.holdings[f"{symbol}_{direction}"]                
                old_contract_value = holding['current_contract_value']
                old_commission = holding['commission']
                new_risk = self.futures_margin[f"{symbol}_{direction}"] / (self.futures_balance - sum(self.futures_margin.values())) 
                
                # 更新持仓数量，这个holding['quantity']是个字典，键为买入价格，值为买入手数
                if current_price not in holding['quantity']:
                    holding['quantity'][current_price] = quantity
                else:
                    holding['quantity'][current_price] += quantity
                # 对于每一笔合约，它们都有各自的买入价格和买入手数，这里需要更新每一笔合约的P&L_point和floating_P&L
                # P&L_point和floating_P&L都是字典，键为买入价格，值为对应P&L_point和floating_P&L的值
                for each_contract_price in holding['quantity'].keys():
                    if direction == 'go_long':
                        holding['P&L_point'][each_contract_price] = current_price - each_contract_price              
                    else:
                        holding['P&L_point'][each_contract_price] = each_contract_price - current_price     
                        print("!!!!")
                    holding['floating_P&L'][each_contract_price] = (holding['P&L_point'][each_contract_price]) * holding['quantity'][each_contract_price] * VolumeMultiple
                holding['last_trade_time'].update({current_price:current_time})
                holding.update({
                    # 'last_trade_time': holding['last_trade_time'],
                    'margin': self.futures_margin[f"{symbol}_{direction}"],
                    'current_price': current_price,
                    'current_contract_value': contract_value + old_contract_value,
                    'total_P&L': sum(holding['floating_P&L'].values()),
                    'risk': new_risk,
                    'commission': commission + old_commission,         
                    'need_add_margin':True if new_risk > 1 else False
                })
            else: # 新增持仓
                self.holdings[f"{symbol}_{direction}"] = {
                    'last_trade_time': {current_price:current_time},  # 最近一次交易时间，键为买入价格，值为最近时间
                    'type' : self.get_instrument_type(symbol), # 商品类型为期货
                    'trading_direction': direction,   # 交易方向。默认多头go_long
                    'quantity': {current_price:quantity},# 用来记录持仓数量，键为买入价格，值为买入手数
                    'margin': Margin,                 # 持仓保证金
                    'current_price': current_price,   # 当前产品指数价格
                    'current_contract_value': contract_value, # 当前持仓合约价值
                    'P&L_point' : {current_price:0},  # 用来记录每笔合约的实时盈亏点数，正数为盈，负数为亏。不论多空头
                    'floating_P&L':{current_price:0}, # 用来记录每笔合约的实时盈亏金额
                    'total_P&L' : 0,                  # 该持仓总盈亏
                    'risk': risk,                     # 持仓风险
                    'commission': commission,         # 手续费
                    'need_add_margin': False,
                    'multiple': VolumeMultiple,
                    'expire_date': ExpireDate,
                    'charge_type': ChargeType,
                    'long_margin_ratio': LongMarginRatio,
                    'short_margin_ratio': ShortMarginRatio,
                    'ChargeOpen': ChargeOpen,
                    'ChargeClose': ChargeClose,
                    'ChargeTodayOpen': ChargeTodayOpen,
                    'ChargeTodayClose': ChargeTodayClose,
                }
                self.futures_margin[f"{symbol}_{direction}"] = Margin
            print(f"以{Margin:.2f}保证金成交开仓{direction}{quantity}手{symbol}，当前期货价格：{current_price}，期货指数合约价值：{self.holdings[f"{symbol}_{direction}"]["current_contract_value"]:.2f}")
            return True
        else:
            # 开仓失败
            print(f"开仓{direction}失败，所需保证金：{Margin}大于当前可用权益：{self.futures_balance-sum(self.futures_margin.values())}")
            return False
    def close_a_futures_position(self,symbol,quantity,direction='go_long',select_to_sell=None):
        symbol_key = f"{symbol}_{direction}" 
        if symbol_key not in self.holdings:
            print(f"{direction}平仓失败，持仓中没有该商品")
            return False
        else:
            if self.holdings[symbol_key]['trading_direction'] != direction:
                print("平仓失败，持仓中该商品不是合约方向相反")
                return False
        holding = self.holdings[symbol_key]
        if quantity > sum(holding['quantity'].values()):
            print("卖出失败，卖出数量大于持仓量")
            return False
        
        count_sell = 0
        current_price = self.get_current_price(symbol)
        holding['current_price'] = current_price
        holding['margin'] = holding['margin'] * (1- (quantity / sum(holding['quantity'].values())))
        self.futures_margin[f"{symbol}_{direction}"] = holding['margin']                

        current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        multiple_factor = 1 if direction == 'go_long' else -1

        for each_contract_price in holding['quantity'].keys():
            holding['P&L_point'][each_contract_price] = (holding['current_price'] - each_contract_price) * multiple_factor              
            holding['floating_P&L'][each_contract_price] = (holding['P&L_point'][each_contract_price]) * holding['quantity'][each_contract_price] * holding['multiple']
            commission = 0 
            while holding['quantity'][each_contract_price] > 0:
                # 更新持仓信息
                old_time = holding['last_trade_time'][each_contract_price]
                holding['last_trade_time'][each_contract_price] = current_time
                holding['quantity'][each_contract_price] -= 1



                if holding['ChargeTodayClose'] != -1: # -1表示没有平仓手续费
                    # 若闭仓和开仓在同一天，则按ChargeTodayClose收费，否则按ChargeClose收费
                    charge_if_close_today = holding['ChargeTodayClose'] if current_time.split(' ')[0] == old_time.split(' ')[0] else holding['ChargeClose']
                    if holding['charge_type'] == 2: # 按费率收费
                        commission = (charge_if_close_today / 10000) * (current_price * holding['multiple'])
                    elif holding['charge_type'] == 1: # 按手数收费
                        commission = charge_if_close_today
                    else:
                        print("未知手续费收费方式")
                holding['commission'] += commission
                self.futures_balance -= commission

                holding['current_contract_value'] -= current_price * holding['multiple']
                # holding['margin'] = holding['current_contract_value'] * holding['long_margin_ratio']
                holding['floating_P&L'][each_contract_price] -= holding['P&L_point'][each_contract_price] * holding['multiple']
                holding['total_P&L'] -= holding['P&L_point'][each_contract_price] * holding['multiple']
                holding['risk'] = holding['margin'] / (self.futures_balance - sum(self.futures_margin.values())) 
                # 更新账户资产信息
                self.futures_balance += holding['P&L_point'][each_contract_price] * holding['multiple']
                self.total_asset = self.outside_balance + self.inside_market_value + self.futures_balance

                if holding['quantity'][each_contract_price] == 0:
                    del holding['quantity'][each_contract_price]
                    del holding['P&L_point'][each_contract_price]
                    del holding['floating_P&L'][each_contract_price]

                count_sell += 1
                if quantity == count_sell: # 卖出数量等于已卖出数量，说明已卖出所有合约，跳出循环
                    break      
            if quantity == count_sell: # 卖出数量等于已卖出数量，说明已卖出所有合约，跳出循环
                break

        if sum(holding['quantity'].values()) == 0:
            del self.holdings[symbol_key]
        print(f"平仓{direction}{quantity}手{symbol},当前期货价格：{current_price}")
        return True
    def buy(self, symbol, quantity):
        """
        买入函数
        :param symbol: 商品代码(str)
        :param quantity: 买入数量(int)
        :return: 买入是否成功(bool)
        """
        current_price = self.get_current_price(symbol)
        
        # volume_multiple = xtdata.get_instrument_detail(symbol, iscomplete=False)['VolumeMultiple']

        total_cost = current_price * quantity
        
        if self.outside_balance >= total_cost:
            # 买入成功
            self.outside_balance -= total_cost
            self.inside_market_value += total_cost
            self.total_asset = self.outside_balance + self.inside_market_value + self.futures_balance
            
            # 更新持仓信息
            current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
            if symbol in self.holdings:
                # 已有持仓，更新信息
                holding = self.holdings[symbol]
                old_quantity = holding['quantity']
                old_buy_amount = holding['buy_amount']
                
                new_quantity = old_quantity + quantity
                new_buy_amount = old_buy_amount + total_cost
                new_buy_price = new_buy_amount / new_quantity
                
                holding.update({
                    'last_trade_time': current_time,
                    'buy_price': new_buy_price,
                    'quantity': new_quantity,
                    'buy_amount': new_buy_amount,
                    'current_price': current_price,
                    'current_market_value': new_quantity * current_price,
                    'profit_loss_ratio': (new_quantity * current_price - new_buy_amount) / new_buy_amount
                })
            else:
                # 新持仓
                self.holdings[symbol] = {
                    'last_trade_time': current_time,
                    'type': self.get_instrument_type(symbol),
                    'buy_price': current_price,
                    'quantity': quantity,
                    'buy_amount': total_cost,
                    'current_price': current_price,
                    'current_market_value': total_cost,
                    'profit_loss_ratio': 0.0
                }
            print(f"以{total_cost:.2f}成交买入{quantity}股{symbol}，成交时市价{current_price}")
            return True
        else:
            # 买入失败，余额不足
            print("买入失败，仓外余额不足")
            return False 
    def sell(self, symbol, quantity):
        """
        卖出函数
        :param symbol: 商品代码(str)
        :param quantity: 卖出数量(int)
        :return: 卖出是否成功(bool)
        """
        if symbol not in self.holdings:
            print("卖出失败，持仓中没有该商品")
            return False
        
        holding = self.holdings[symbol]
        if quantity > holding['quantity']:
            print("卖出失败，卖出数量大于持仓量")
            return False
        
        # 卖出成功
        current_price = self.get_current_price(symbol)
        total_proceeds = current_price * quantity
        
        # 更新账户信息
        self.inside_market_value -= total_proceeds
        self.outside_balance += total_proceeds
        self.total_asset = self.outside_balance + self.inside_market_value + self.futures_balance
        
        # 更新持仓信息
        current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        remaining_quantity = holding['quantity'] - quantity
        if remaining_quantity > 0:
            # 还有剩余持仓
            holding.update({
                'last_trade_time': current_time,
                'quantity': remaining_quantity,
                'current_price': current_price,
                'current_market_value': remaining_quantity * current_price,
                'profit_loss_ratio': (remaining_quantity * current_price + total_proceeds - holding['buy_amount']) / holding['buy_amount']
            })
        else:
            # 全部卖出，移除持仓
            del self.holdings[symbol]
        print(f"以{total_proceeds:.2f}成交卖出{quantity}股{symbol},成交时市价{current_price}")
        return True
    def transfer_to_futures_position(self, amount):
        """
        从仓外余额转入期货仓
        :param amount: 转入金额 (float)
        :return: 是否成功 (bool)
        """
        if not hasattr(self, 'futures_balance'):
            self.futures_balance = 0.0
        if amount <= 0:
            print("转入金额必须大于0")
            return False
        if self.outside_balance >= amount:
            self.outside_balance -= amount
            self.futures_balance += amount
            print(f"从仓外转入 {amount:.2f} 至期货仓，当前期货仓余额: {self.futures_balance:.2f}")
            return True
        else:
            print("仓外余额不足，转入失败")
            return False
    def transfer_out_futures_position(self, amount):
        """
        从期货仓转出到仓外余额
        :param amount: 转出金额 (float)
        :return: 是否成功 (bool)
        """
        if not hasattr(self, 'futures_balance'):
            self.futures_balance = 0.0
        if amount <= 0:
            print("转出金额必须大于0")
            return False
        if self.futures_balance >= amount:
            self.futures_balance -= amount
            self.outside_balance += amount
            print(f"从期货仓转出 {amount:.2f} 至仓外，当前期货仓余额: {self.futures_balance:.2f}")
            return True
        else:
            print("期货仓余额不足，转出失败")
            return False
    def clear_inventory(self,stock = 1,fund = 1,futures = 1): # 一键清仓,默认为1,表示该种类商品全部卖出
        info = ""

        for symbol in list(self.holdings.keys()):
            if self.holdings[symbol]['type'] == 'STOCK' and stock == 1:
                self.sell(symbol,self.holdings[symbol]['quantity'])
            elif self.holdings[symbol]['type'] == 'FUND' and fund == 1:
                self.sell(symbol,self.holdings[symbol]['quantity']) 
            elif self.holdings[symbol]['type'] == 'FUTURES' and futures == 1:
                self.close_a_futures_position(symbol.split('_')[0], sum(self.holdings[symbol]['quantity'].values()), self.holdings[symbol]['trading_direction'])
        if stock == 1:
            info += f"已卖出所有股票."
        if fund == 1:
            info += f"已卖出所有基金."
        if futures == 1:
            info += f"已卖出所有期货.\n"
        print(info)
        return True
    def __str__(self):
        """
        打印账户信息
        :return: 账户信息字符串
        """
        info = f"账号: {self.account_id}\n"
        info += f"仓外余额: {self.outside_balance:.2f}\n"        

        holdings_codes = list({key.split('_')[0] for key in self.holdings})
        # current_holdings_lastPrice = self.get_current_price(list(self.holdings.keys()))
        current_holdings_lastPrice = self.get_current_price(holdings_codes)

        stock_market_value = 0.0
        futures_market_PL = 0.0
        for custom in self.holdings.keys():
            if len(custom.split('_')) != 3: # 为期货交易，期货交易的键命名为 商品代码_go_long或者商品代码_go_short
            # if self.holdings[custom]['type'] != 'FUTURES': # 非期货持仓更新
                self.holdings[custom]['current_price'] = current_holdings_lastPrice[custom]
                self.holdings[custom]['current_market_value'] = self.holdings[custom]['current_price'] * self.holdings[custom]['quantity']
                self.holdings[custom]['profit_loss_ratio'] = (self.holdings[custom]['current_market_value'] - self.holdings[custom]['buy_amount']) / self.holdings[custom]['buy_amount']
                stock_market_value += self.holdings[custom]['current_market_value']
            else:                           # 期货持仓跟新
                multiple_direction = 1 # 乘数方向,用来控制多空盈亏点数
                if custom.split('_')[-1] == 'long':
                    multiple_direction = 1
                else:
                    multiple_direction = -1
                self.holdings[custom]['current_price'] = current_holdings_lastPrice[custom.split('_')[0]]
                self.holdings[custom]['current_contract_value'] = self.holdings[custom]['current_price'] * sum(self.holdings[custom]['quantity'].values()) * self.holdings[custom]['multiple']
                for each_contract_price in self.holdings[custom]['quantity'].keys():
                    self.holdings[custom]['P&L_point'][each_contract_price] = multiple_direction * (self.holdings[custom]['current_price'] - each_contract_price)              
                    self.holdings[custom]['floating_P&L'][each_contract_price] = (self.holdings[custom]['P&L_point'][each_contract_price]) * self.holdings[custom]['quantity'][each_contract_price] * self.holdings[custom]['multiple']
                self.holdings[custom]['total_P&L'] = sum(self.holdings[custom]['floating_P&L'].values())
                self.holdings[custom]['risk'] = self.futures_margin[custom] / (self.futures_balance + self.holdings[custom]['total_P&L'])
                futures_market_PL += self.holdings[custom]['total_P&L']
        self.inside_market_value = stock_market_value + futures_market_PL
        self.total_asset = self.inside_market_value + self.outside_balance + self.futures_balance
        info += f"仓内市值 + 期货盈亏: {self.inside_market_value:.3f}\n"
        info += f"期货仓余额: {self.futures_balance:.3f}\n"
        info += f"被冻结保证金: {sum(self.futures_margin.values()):.3f}\n"

        info += f"总资产: {self.total_asset:.3f}\n"
        info += "持仓:\n"
        if not self.holdings:
            info += "  无持仓\n"
        else:
            for symbol, holding in self.holdings.items():
                if holding['type'] != 'FUTURES':
                    info += f"  {holding['type']} {symbol}:\n"
                    info += f"    最近交易时间: {holding['last_trade_time']}\n"
                    info += f"    买入时市价: {holding['buy_price']:.3f}\n"
                    info += f"    持仓量: {holding['quantity']}\n"
                    info += f"    买入金额: {holding['buy_amount']:.2f}\n"
                    info += f"    当前市价: {holding['current_price']:.3f}\n"
                    info += f"    当前仓位市值: {holding['current_market_value']:.3f}\n"
                    info += f"    当前盈亏比: {holding['profit_loss_ratio']:.4f}\n"
                else:
                    info += f"  {holding['type']} {symbol}:\n"
                    info += f"    最近交易时间: {holding['last_trade_time']}\n"
                    info += f"    交易方向: {holding['trading_direction']}\n"

                    info += f"    持仓量: {holding['quantity']}\n"
                    info += f"    冻结保证金: {holding['margin']:.2f}\n"
                    info += f"    当前指数价格: {holding['current_price']:.3f}\n"
                    info += f"    当前合约市价: {holding['current_contract_value']:.3f}\n"
                    info += f"    当前盈亏: {holding['total_P&L']:.3f}\n"
                    info += f"    当前风险度: {holding['risk']:.4f}\n"
        return info
    def auto_trade_stock(self, stock_code, buy_price, stop_loss_price, take_profit_price, quantity, max_trades):
        """
        自动监控交易函数
        :param stock_code: 商品代码(str)
        :param buy_price: 监控买入价(float)
        :param stop_loss_price: 止损卖出价(float)
        :param take_profit_price: 止盈卖出价(float)
        :param quantity: 买入量(int)
        :param max_trades: 交易次数(int)
        :return: 订阅ID
        """
        # 初始化交易状态
        self._trade_status = {
            'stock_code': stock_code,
            'buy_price': buy_price,
            'stop_loss_price': stop_loss_price,
            'take_profit_price': take_profit_price,
            'quantity': quantity,
            'max_trades': max_trades,
            'trade_count': 0,
            'subscribe_id': None,
            'is_holding': False
        }
        # 定义回调函数
        def on_data(datas):
            if stock_code not in datas:
                return
            # 获取实时数据
            data = datas[stock_code][0]
            last_price = data['lastPrice']
            ask_prices = data['askPrice']  # 委卖价
            bid_prices = data['bidPrice']  # 委买价
            ask_vols = data['askVol']      # 委卖量
            bid_vols = data['bidVol']      # 委买量
            
            # 打印实时行情
            current_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f"{current_time}")
            print(f"股票代码为{stock_code}, 目前市价为{last_price}")
            
            # 检查交易次数
            if self._trade_status['trade_count'] >= max_trades:
                print(f"已达到最大交易次数{max_trades}，停止自动交易")
                self.stop_auto_trade_stock()
                return
            
            # 检查买入条件
            if not self._trade_status['is_holding']:
                if (last_price <= buy_price and last_price > stop_loss_price):
                    # 执行买入
                    success = self.buy(stock_code, quantity)
                    if success:
                        self._trade_status['is_holding'] = True
                        print(f"自动交易第{self._trade_status['trade_count'] + 1}次: 买入成功")
                    else:
                        print(f"仓外余额不做，买入失败")
                        self.stop_auto_trade()
                        return
            # 检查卖出条件
            else:                        
                if (last_price <= stop_loss_price or last_price >= take_profit_price):
                    # 执行卖出
                    success = self.sell(stock_code, quantity)
                    if success:
                        self._trade_status['is_holding'] = False
                        print(f"自动交易第{self._trade_status['trade_count'] + 1}次: 卖出成功")
                        self._trade_status['trade_count'] += 1
        # 订阅行情
        subscribe_id = xtdata.subscribe_quote(stock_code, period='tick', callback=on_data)
        self._trade_status['subscribe_id'] = subscribe_id
        print(f"开始自动交易监控，订阅ID: {subscribe_id}")
        print(f"监控股票: {stock_code}")
        print(f"买入价格: {buy_price}, 止损价格: {stop_loss_price}, 止盈价格: {take_profit_price}")
        print(f"买入量: {quantity}, 最大交易次数: {max_trades}")
        
        return subscribe_id 
    def stop_auto_trade_stock(self):
        """
        停止自动交易
        """
        if hasattr(self, '_trade_status') and self._trade_status['subscribe_id']:
            subscribe_id = self._trade_status['subscribe_id']
            xtdata.unsubscribe_quote(subscribe_id)
            print(f"已取消订阅，订阅ID: {subscribe_id}")
            self._trade_status['subscribe_id'] = None
        else:
            print("没有正在进行的自动交易")
    def spot_futures_arbitrage(self,futures_symbol,index_code,quantity,basis_threshold,upper_basis_limit,lower_basis_limit,direction = 'go_long'):
        # 期现套利，目前只能做多基差（做多现货，做空期货）
        I_base,P_base_dict,weight_dict = read_last_trade_day_data_for_prepare_to_get_realtime_basis(index_code)
        reverse_direction = 'go_short' if direction == 'go_long' else 'go_long'
        print("实时监控套利机会")
        while True:
            real_time_basis, stocks_buy_amount = get_realtime_computing_index(index_code,I_base,P_base_dict,weight_dict,futures_symbol)
            if real_time_basis >= basis_threshold and real_time_basis < upper_basis_limit:  # 当基差过大时候,出现正向期限套利机会
                self.open_a_futures_position(futures_symbol,quantity,reverse_direction)
                for stock_code in stocks_buy_amount:
                    self.buy(stock_code,stocks_buy_amount[stock_code])
                print("开始正向套利,做多基差")
                break
            time.sleep(0.1)
        while True:
            real_time_basis,stocks_buy_amount = get_realtime_computing_index(index_code,I_base,P_base_dict,weight_dict,futures_symbol)
            if real_time_basis > upper_basis_limit: # 回归正常水平基差,止盈基差
                self.close_a_futures_position(futures_symbol,quantity,reverse_direction)
                self.clear_inventory(stock = 1,fund = 0,futures = 0)
                print("到达止盈基差点，结束正向套利")
                break
            if real_time_basis < lower_basis_limit: # 超底基差,止损基差
                self.close_a_futures_position(futures_symbol,quantity,reverse_direction)
                self.clear_inventory(stock = 1,fund = 0,futures = 0)
                print("到达止损基差点，结束正向套利")
                break
            time.sleep(0.1)


In [ ]:
期货高手 = TradingAccount("kmgasd1",5000000)
期货高手.transfer_to_futures_position(1000000)
print(期货高手)

从仓外转入 1000000.00 至期货仓，当前期货仓余额: 1000000.00
***** xtdata连接成功 2026-03-26 11:30:28*****
服务信息: {'tag': 'xtquant', 'version': '1.0'}
服务地址: 127.0.0.1:58665
数据路径: d:\test_AI_project\xtquant_tushare\srv\xuntoudata\data\datadir
设置xtdata.enable_hello = False可隐藏此消息

以183009.60保证金成交1手IM2604.IF，合约价值：1525080.00
账号: kmbzg144
仓外余额: 1000000.00
仓内市值 + 期货盈亏: -0.000
期货仓余额: 1000000.000
被冻结保证金: 183009.600
总资产: 2000000.000
持仓:
  FUTURES IM2604.IF_go_short:
    最近交易时间: 2026-03-26 11:28:30
    交易方向: go_short
    持仓量: {7625.4: 1}
    冻结保证金: 183009.60
    当前指数价格: 7625.400
    当前合约市价: 1525080.000
    当前盈亏: -0.000
    当前风险度: 0.1830



In [ ]:
basis_threshold = 63
upper_basis_limit = basis_threshold + 5
lower_basis_limit = basis_threshold - 3
futures_index_pair = ("IF2606.IF","000300.SH")
futures_index_pair = ("IM2604.IF","000852.SH")
期货高手.spot_futures_arbitrage(futures_symbol="IF2606.IF",index_code="000300.SH",quantity=1,basis_threshold=basis_threshold,upper_basis_limit=upper_basis_limit,lower_basis_limit=lower_basis_limit,direction="go_long")

账号: kmbzg144
仓外余额: 1000000.00
仓内市值 + 期货盈亏: 0.000
期货仓余额: 1000120.000
被冻结保证金: 0.000
总资产: 2000120.000
持仓:
  无持仓



In [ ]:
print(期货高手)

账号: kmbzg144
仓外余额: 1000000.00
仓内市值: 0.000
期货仓余额: 1000000.000
总资产: 2000000.000
持仓:
  无持仓

